# Reproducibility and estimator validation

This notebook covers two things:
- **Reproducibility:** every run can be regenerated bit-for-bit from its
  manifest.
- **Validation:** because every generator's parameters are known, we can
  measure the bias and CI coverage of each estimator.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pcc_vizforge.plots.style import apply_matplotlib_style

apply_matplotlib_style()
SEED = 20240101

In [2]:
import json, tempfile
from pathlib import Path
from pcc_vizforge.experiments import run_experiment, verify_run

workdir = Path(tempfile.mkdtemp())
run = run_experiment("quakes", seed=7, overrides=["data_generation.b_value=0.9"], output_dir=workdir, formats=("png",))
print(sorted(p.name for p in run.run_dir.iterdir()))
manifest = json.loads((run.run_dir / "manifest.json").read_text())
{k: manifest[k] for k in ("run_id", "seed", "config_sha256", "outputs")}

['config.yaml', 'data.csv', 'figures', 'manifest.json', 'metrics.json']


{'run_id': '55f04a103268',
 'seed': 7,
 'config_sha256': '25e8370784fedf991d275950c6bcfec4cdc72433c7830d90ba0b7b5d6a982f14',
 'outputs': {'config.yaml': 'e53ca9d358e90be32bc8e0e9fd623658fb4506f3c3015242717bde2907c4a0dd',
  'data.csv': 'e4a50448fbf496240f2b0d5ef597608e63185ebed2b6ee4d93a6cddc994f1600',
  'figures/aftershock_decay.png': 'b9ca81b0f8f9cbb7227a77e96236fa0e7a1b2e14b070309f9defdb4c71147047',
  'figures/gutenberg_richter.png': '00ccd00853f69acfa49b20ad337cf734f2264cc16e243e05b12e0b64c372befe',
  'metrics.json': '9a1ed6733cecac41e6016231df9136c0379f085313650aeb35d74f196d53d977'}}

In [3]:
verify_run(run.run_dir)

{'run_id': '55f04a103268',
 'domain': 'quakes',
 'seed': 7,
 'reproduced': True,
 'integrity': {'config.yaml': True,
  'data.csv': True,
  'figures/aftershock_decay.png': True,
  'figures/gutenberg_richter.png': True,
  'metrics.json': True},
 'recorded_versions': {'matplotlib': '3.11.2',
  'numpy': '2.5.3',
  'pandas': '3.0.6',
  'plotly': '7.1.0',
  'pyyaml': '6.0.3',
  'scipy': '1.18.1'}}

Tampering with a stored file is detected:

In [4]:
(run.run_dir / "metrics.json").write_text("{}")
verify_run(run.run_dir)["integrity"]

{'config.yaml': True,
 'data.csv': True,
 'figures/aftershock_decay.png': True,
 'figures/gutenberg_richter.png': True,
 'metrics.json': False}

## Monte Carlo validation

Each study simulates independent replicates on spawned RNG streams (100 here; the published tables in `docs/validation.md` use 500).

In [5]:
from pcc_vizforge.experiments.validation import run_study

for name in ("b_value", "power_law", "msd_exponent"):
    res = run_study(name, n_replicates=100, seed=SEED)
    print(f"\n{res.description}")
    display(res.summary.drop(columns=["n_replicates"]).round(4))


Aki-Utsu b-value MLE (true b = 1) vs catalogue size


,n_events,truth,mean_estimate,bias,bias_mcse,rmse,coverage_95,coverage_mcse
0,50,1.0,1.0163,0.0163,0.0168,0.1677,0.92,0.0271
1,100,1.0,1.0094,0.0094,0.0117,0.1167,0.89,0.0313
2,250,1.0,1.0038,0.0038,0.0066,0.0658,0.94,0.0237
3,500,1.0,1.0030,0.0030,0.0047,0.0471,0.96,0.0196
4,1000,1.0,1.0030,0.0030,0.0035,0.0349,0.92,0.0271
5,2500,1.0,1.0022,0.0022,0.0021,0.0209,0.91,0.0286



Discrete power-law MLE (x_min = 1, n = 1000)


,alpha,truth,mean_estimate,bias,bias_mcse,rmse,coverage_95,coverage_mcse
0,1.8,1.8,1.8029,0.0029,0.0029,0.0295,0.91,0.0286
1,2.2,2.2,2.2045,0.0045,0.0046,0.0456,0.92,0.0271
2,2.6,2.6,2.6077,0.0077,0.0065,0.0656,0.91,0.0286
3,3.0,3.0,3.0070,0.0070,0.0085,0.0850,0.92,0.0271



Walk-bootstrap MSD exponent for fBm (true α = 2H)


,hurst,truth,mean_estimate,bias,bias_mcse,rmse,coverage_95,coverage_mcse
0,0.25,0.5,0.5035,0.0035,0.0033,0.0327,0.95,0.0218
1,0.50,1.0,1.0047,0.0047,0.0036,0.0358,0.96,0.0196
2,0.75,1.5,1.5052,0.0052,0.0039,0.0396,0.97,0.0171


In [6]:
from pcc_vizforge.plots import diagnostics as D

dice = run_study("dice_gof", n_replicates=300, seed=SEED)
display(dice.summary.round(3))
D.pvalue_calibration_figure(dice.replicates.query("n_dice == 3")["p_value"], label="χ² GOF, 3 dice");

,n_dice,n_replicates,size_at_5pct,size_mcse,ks_uniform_pvalue
0,1,300,0.070,0.015,0.104
1,2,300,0.043,0.012,0.427
2,3,300,0.037,0.011,0.012
3,5,300,0.060,0.014,0.383
